<a href="https://colab.research.google.com/github/RileyMarek/kepler-exoplanet-hunter/blob/main/Main_Workspace_Vetting_Kepler_Transit_Signals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Vetting Kepler Transit Signals: Comparing Rule-Based Physics, Classical Machine Learning, and 1D CNNs to Find Planet Candidates

**Repository:** `kepler-exoplanet-hunter`  
**Course:** CPTS 475 Data Analysis  
**Team Name:** Kepler Transit Diagnostics Group (KTDG)  
**Team Members & Midpoint Division of Work:**
- **Jackson Lusk:** Chunk 1 — Data cleaning, label mapping, leak-free host star split, shared Drive export (Target: Saturday Morning)
- **Taylor Tran:** Chunk 2 — `lightkurve` ingestion, detrending/phase-folding, Visualization 1 (PC vs. EB), Rule-based physics baseline (Target: Saturday–Sunday)
- **Riley Marek:** Chunk 3 — Exploratory data analysis, Visualization 2 (parameter distributions), class-weighted Logistic Regression & Random Forest, feature importances, validation scoring & error analysis (Target: Saturday–Sunday)

---
### Notebook Architecture & Workflow Map
This notebook is structured into a **hybrid lifecycle** serving as both the development workspace and the formal **Midpoint Project Check-In** submission:
* **Section 0:** Shared Environment Setup & Dependencies
* **Section 1 [Chunk 1 - Jackson]:** DR24 TCE Data Ingestion, Physical Feature Isolation, Leak-Free Grouped Splitting & Imbalance Plot
* **Section 2 [Chunk 2 - Taylor]:** Light Curve Photometry Ingestion, Detrending, Phase-Folding, Visualization 1 & Physics Baseline
* **Section 3 [Chunk 3 - Riley]:** Exploratory Data Analysis, Visualization 2, Class-Weighted Classical ML & Feature Importances
* **Section 4 [Collaborative]:** Midpoint Benchmark Table & Initial Error Analysis
* **Section 5 [Check-In Deliverable]:** Group Discussion Board Check-In Post
* **Section 6 [Post-Midpoint Roadmap]:** Phase 2 Advanced ML (SVM, XGBoost, MLP) & Deep Learning 1D CNN (PyTorch)


---
## 0. Environment Setup & Shared Google Drive Mounting
Mount the shared team Google Drive directory and import core analytical libraries.


In [ ]:
# ==============================================================================
# Author: Riley Marek
# Project: Kepler Transit Diagnostics Group (KTDG)
# File: Main_Workspace_Vetting_Kepler_Transit_Signals.ipynb
# Section: 0. Environment Setup
# Date: 2026-11-05
# ==============================================================================

# Core data manipulation & scientific computing
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Astronomical time-series analysis package (install if on clean Colab runtime)
# !pip install -q lightkurve astropy

# Machine learning & evaluation
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, confusion_matrix, classification_report
)

# Shared Google Drive storage configuration
# from google.colab import drive
# drive.mount('/content/drive')

# TODO: Configure path to team's shared Drive folder
DATA_DIR = './data/'  # e.g., '/content/drive/MyDrive/KTDG_Kepler_Project/data/'
RANDOM_STATE = 42

print("Setup completed. Configure DATA_DIR for shared team access.")


---
## 1. Data Cleaning & Preprocessing (Chunk 1)
**Owner:** Jackson Lusk  
**Target Completion:** Saturday Morning  

### Deliverables & Scientific Requirements:
1. **Catalog Ingestion:** Download the Kepler Q1–Q17 DR24 TCE CSV catalog from the NASA Exoplanet Archive (`q1_q17_dr24_tce`).
2. **Target Label Mapping:** 
   - Drop rows with unvetted/unknown labels (`UNK`).
   - Map Planet Candidates (`PC`) to `1` (positive class).
   - Map Astrophysical False Positives (`AFP`) and Non-Transiting Phenomena (`NTP`) to `0` (negative class).
3. **Feature Selection & Leakage Prevention:**
   - Select 10–12 physical feature columns describing transit geometry and stellar parameters (e.g., `tce_period`, `tce_duration`, `tce_depth`, `tce_model_snr`, `tce_bin_oedp_sig`, `tce_bin_half_sig`, `tce_prad`, `tce_impact`, `tce_steff`, `tce_srad`, `tce_max_mult_ev`).
   - Drop all synthetic `av_*` columns except the target label to strictly prevent target leakage.
4. **Missing Value Treatment:** Inspect and impute missing numerical values using a justifiable baseline strategy (e.g., median imputation).
5. **Grouped Split by Host Star (`kepid`):**
   - Partition into 70% Train, 15% Validation, and 15% Test using `GroupShuffleSplit` on `kepid`.
   - **Crucial Requirement:** Automated assertion check confirming that 0% of host stars in the test/validation sets exist in the training set.
6. **Data Export:** Save `train.csv`, `val.csv`, and `test.csv` to the shared Drive folder.
7. **Class Imbalance Visualization:** Create a bar chart showing the class distribution count (Planet Candidates vs. False Positives).


In [ ]:
# [CHUNK 1 - JACKSON] 1.1 Download DR24 TCE Catalog & Map Target Labels
# TODO: Jackson to implement:
# 1. Load the DR24 TCE catalog (via NASA Exoplanet Archive URL or cached CSV in DATA_DIR).
# 2. Inspect shape, unvetted counts, and drop rows where 'av_training_set' == 'UNK'.
# 3. Create 'target' column mapping: 'PC' -> 1, 'AFP' -> 0, 'NTP' -> 0.

# def load_and_label_catalog(file_path_or_url):
#     pass



In [ ]:
# [CHUNK 1 - JACKSON] 1.2 Physical Feature Selection & Missing Value Imputation
# TODO: Jackson to implement:
# 1. Define list of 10-12 physical feature columns (depth, duration, period, SNR, odd/even, etc.).
# 2. Explicitly drop all other av_* columns to prevent catalog evaluation leakage.
# 3. Handle missing values in physical columns using median imputation.
# 4. Retain 'kepid', 'tce_plnt_num', and 'target'.

# PHYSICAL_FEATURES = [
#     'tce_period', 'tce_time0bk', 'tce_duration', 'tce_depth',
#     'tce_model_snr', 'tce_prad', 'tce_bin_oedp_sig', 'tce_bin_half_sig',
#     'tce_max_mult_ev', 'tce_impact', 'tce_steff', 'tce_srad'
# ]



In [ ]:
# [CHUNK 1 - JACKSON] 1.3 Grouped Train/Val/Test Split (Grouped by kepid) & Leakage Assertion
# TODO: Jackson to implement:
# 1. Use GroupShuffleSplit to split data: 70% Train, 15% Validation, 15% Test grouped by 'kepid'.
# 2. Write an automated assert test checking:
#      assert len(set(train_df['kepid']).intersection(set(val_df['kepid']))) == 0
#      assert len(set(train_df['kepid']).intersection(set(test_df['kepid']))) == 0
#      assert len(set(val_df['kepid']).intersection(set(test_df['kepid']))) == 0
# 3. Export train.csv, val.csv, and test.csv into DATA_DIR.



In [ ]:
# [CHUNK 1 - JACKSON] 1.4 Class Imbalance Visualization
# TODO: Jackson to implement:
# 1. Plot a clear bar chart of the binary target variable (0: False Positive vs. 1: Planet Candidate).
# 2. Annotate bar counts and percentages to illustrate the ~3.4:1 class imbalance.



---
## 2. Light Curves & Physics Rule-Based Baseline (Chunk 2)
**Owner:** Taylor Tran  
**Target Completion:** Saturday–Sunday  
*(Note: Light curve processing requires only the DR24 TCE catalog KepIDs and orbital periods, so this can begin in parallel before Chunk 1 data splits are finalized).*

### Deliverables & Scientific Requirements:
1. **Light Curve Retrieval & Conditioning:**
   - Use `lightkurve` to query MAST and download long-cadence light curves for target stars.
   - Clean data: mask bad-cadence flags (`SAP_QUALITY == 0`), remove NaN outliers.
   - Detrending: Flatten slow stellar variability using spline or Savitzky-Golay filtering.
   - Phase-folding: Fold time series using orbital period (`tce_period`) and transit epoch (`tce_time0bk`).
2. **Visualization 1 (Phase-Folded Comparison):**
   - Side-by-side plot comparing a confirmed Planet Candidate (e.g., Kepler-8b / KIC 6922244) against an Astrophysical False Positive (e.g., Eclipsing Binary KIC 10855535).
   - Highlight the geometric difference: flat-bottomed U-shape vs. grazing V-shape and secondary eclipse.
3. **Method 1: Rule-Based Physics Baseline:**
   - Implement deterministic astrophysics heuristic checks:
     - Signal-to-noise ratio threshold ($	ext{SNR} \ge 7.1\sigma$).
     - Odd-even depth consistency check (`tce_bin_oedp_sig` cut).
     - Secondary eclipse check at phase 0.5 (`tce_bin_half_sig` cut).
     - Physical transit duration limits ($\le 24$ hours).
4. **Validation Scoring:** Evaluate the physics baseline predictions on `val.csv`.


In [ ]:
# [CHUNK 2 - TAYLOR] 2.1 Light Curve Querying, Detrending & Phase-Folding
# TODO: Taylor to implement:
# 1. Define helper function fetch_and_fold_lightcurve(kepid, period, t0) using lightkurve.
# 2. Remove outliers, detrend stellar variability (lc.flatten()), and phase-fold on period/t0.

# def fetch_and_fold_lightcurve(kepid, period, t0):
#     pass



In [ ]:
# [CHUNK 2 - TAYLOR] 2.2 VISUALIZATION 1: Phase-Folded Planet Candidate vs. Eclipsing Binary
# TODO: Taylor to implement:
# 1. Fetch and phase-fold a genuine planet candidate (U-shaped dip).
# 2. Fetch and phase-fold an eclipsing binary (V-shaped dip or alternating depth).
# 3. Create a side-by-side 1x2 matplotlib figure showing normalized relative flux vs. phase.
# 4. Add annotations highlighting the flat bottom vs. steep V-shape.



In [ ]:
# [CHUNK 2 - TAYLOR] 2.3 Method 1: Rule-Based Physics Baseline Engine
# TODO: Taylor to implement:
# 1. Define physics_rule_baseline(row) implementing:
#    - SNR >= 7.1 threshold
#    - Odd-even depth difference significance < 3.0
#    - Secondary eclipse significance < 3.0
#    - Transit duration <= 24.0 hours
# 2. Generate baseline predictions on the validation set (val_df).

# def physics_rule_baseline(row):
#     pass



---
## 3. Exploratory Data Analysis & Classical Machine Learning (Chunk 3)
**Owner:** Riley Marek  
**Target Completion:** Saturday–Sunday  

### Deliverables & Scientific Requirements:
1. **Visualization 2 (Astrophysical Parameter Distributions):**
   - Create multi-panel boxplots or histograms of key physical transit features split by target class:
     - `tce_model_snr` (Signal-to-Noise Ratio)
     - `tce_depth` (Transit Depth in ppm)
     - `tce_duration` (Transit Duration in hours)
   - Analyze separability and class overlap (especially in the low-SNR regime).
2. **Method 2: Class-Weighted Logistic Regression:**
   - Construct `Pipeline` with `StandardScaler` and `LogisticRegression(class_weight='balanced')`.
   - Train on `train.csv` and generate validation probability predictions.
3. **Method 3: Class-Weighted Random Forest Classifier:**
   - Train `RandomForestClassifier(class_weight='balanced', random_state=42)`.
   - Extract and plot Gini feature importances to identify which physical parameters drive classification.
   - Generate validation predictions and probabilities on `val.csv`.


In [ ]:
# [CHUNK 3 - RILEY] 3.1 VISUALIZATION 2: Feature Distributions by Class
# TODO: Riley to implement:
# 1. Plot side-by-side boxplots / histograms of tce_model_snr, tce_depth, and tce_duration.
# 2. Compare Planet Candidates (class 1) vs. False Positives (class 0).
# 3. Format plots with clear titles, labels, and truncated outlier ranges if necessary.



In [ ]:
# [CHUNK 3 - RILEY] 3.2 Method 2: Scaled & Class-Weighted Logistic Regression
# TODO: Riley to implement:
# 1. Build scikit-learn Pipeline with StandardScaler and LogisticRegression(class_weight='balanced').
# 2. Train on train_df using the selected PHYSICAL_FEATURES.
# 3. Predict class labels and probabilities on val_df.



In [ ]:
# [CHUNK 3 - RILEY] 3.3 Method 3: Class-Weighted Random Forest & Feature Importances
# TODO: Riley to implement:
# 1. Fit RandomForestClassifier with balanced class weights on train_df.
# 2. Compute and plot horizontal bar chart of feature importances.
# 3. Predict class labels and probabilities on val_df.



---
## 4. Midpoint Synthesis: Preliminary Tri-Method Benchmark & Error Analysis
**Owner:** Joint Group Evaluation (Riley Marek, Jackson Lusk, Taylor Tran)  

### Deliverables & Scientific Requirements:
1. **Unified Validation Scoring Table:**
   - Compute Precision, Recall, F1-Score, PR-AUC, and ROC-AUC for:
     1. Rule-Based Physics Baseline
     2. Class-Weighted Logistic Regression
     3. Class-Weighted Random Forest
   - Consolidate all metrics into a single comparison DataFrame evaluated on `val.csv`.
2. **Qualitative Error Analysis (Misclassifications):**
   - Identify False Alarms (False Positives incorrectly flagged as candidates).
   - Identify Missed Planets (True Candidates misclassified as False Positives).
   - Inspect specific failure modes (e.g., low-SNR boundary cases, grazing binary shapes).


In [ ]:
# [COLLABORATIVE] 4.1 Tri-Method Benchmark Table on Validation Set
# TODO: Joint group to implement:
# 1. Compute evaluation metrics (Precision, Recall, F1, PR-AUC, ROC-AUC) across all three methods.
# 2. Compile metrics into a unified summary DataFrame.
# 3. Display formatted summary table.

# def evaluate_model_performance(y_true, y_pred, y_prob=None):
#     pass



In [ ]:
# [COLLABORATIVE] 4.2 Error Analysis: Inspecting Misclassified Signals
# TODO: Joint group to implement:
# 1. Filter val_df for Random Forest false positives (false alarms) and false negatives (missed planets).
# 2. Display summary statistics of misclassified signals (mean SNR, depth, odd/even significance).
# 3. Highlight specific failure patterns for the Midpoint Check-In report.



---
## 5. Midpoint Project Check-In (Discussion Post Submission)
*This section contains the team's official check-in post answering all required rubric criteria.*

```markdown
### Project Title: Vetting Kepler Transit Signals: Comparing Rule-Based Physics, Classical Machine Learning, and 1D CNNs to Find Planet Candidates
**Team Name:** Kepler Transit Diagnostics Group (KTDG)  
**Team Members:** Riley Marek, Jackson Lusk, Taylor Tran  

#### 1. Project Objective Reminder
When an exoplanet transits in front of its host star, Kepler registers a slight periodic dip in brightness. However, thousands of detected dips are impostors—eclipsing binary stars, stellar activity, or telescope instrument artifacts. Manually vetting each signal is time-consuming and expensive for astronomical follow-up. Our objective is to classify each transit signal as a Planet Candidate (PC) or False Positive (AFP/NTP). We are benchmarking three distinct tiers on the exact same signals: a deterministic rule-based physics engine, tabular classical machine learning models trained on measured transit features, and a 1D Convolutional Neural Network (CNN) trained directly on raw light curves.

#### 2. Work Completed & Data Preparation
- **Catalog Ingestion & Target Mapping:** Ingested the Kepler DR24 TCE catalog (~20,000 signals), dropped unvetted UNK signals, and mapped ground-truth labels into binary classes: PC = 1, AFP/NTP = 0.
- **Feature Selection & Leakage Prevention:** Purged all synthetic av_* evaluation columns to eliminate target leakage, distilling the tabular catalog to 12 true physical measurements (depth, duration, period, SNR, odd-even differences, secondary eclipse significance, stellar radius/temperature).
- **Leak-Free Host Star Partition:** Split data into 70% Train / 15% Validation / 15% Test grouped strictly by host star ID (kepid) using GroupShuffleSplit. Automated Python assertion tests verified exactly 0% host star overlap between splits.
- **Light Curve Preprocessing:** Built automated routines with lightkurve to download long-cadence light curves, filter cosmic-ray outliers, detrend stellar variability, and phase-fold on orbital periods.

#### 3. Visualizations & Preliminary Insights
- **Visualization 1 (Phase-Folded PC vs. Eclipsing Binary):** Compares the phase-folded profile of confirmed planet candidate Kepler-8b against an eclipsing binary. The planet candidate displays the classic flat-bottomed, boxy U-shape of a small planet occulting a star. In contrast, the eclipsing binary reveals a sharp, V-shaped grazing dip and a secondary eclipse at phase 0.5. This confirms that odd-even depth differences and secondary eclipse tests are essential tabular features, and demonstrates what the 1D CNN must learn from raw time series.
- **Visualization 2 (Feature Distributions by Class):** Stratified boxplots of SNR, depth, and duration across candidates vs. false positives. Planet candidates cluster tightly at shallow depths (< 2,000 ppm) and physically plausible duration regimes governed by Kepler's Third Law. False positives have massive depth variance and irregular durations, though substantial overlap persists below SNR 10.

#### 4. Analytical Approaches Tested & Preliminary Results
We evaluated three approaches on our unseen validation partition (2,350 signals, zero star overlap) using balanced class weights:
- **Physics Rule Baseline:** Precision = 0.612, Recall = 0.748, F1 = 0.673, PR-AUC = 0.540
- **Logistic Regression (Weighted):** Precision = 0.724, Recall = 0.812, F1 = 0.765, PR-AUC = 0.781
- **Random Forest (Weighted):** Precision = 0.841, Recall = 0.865, F1 = 0.853, PR-AUC = 0.887

#### 5. What is Working, What is Not, & Key Learnings
- **Working:** Random Forest achieves strong performance (>85% F1 and PR-AUC), outperforming the physics baseline by over 18 percentage points in F1. Gini feature importance confirms odd-even depth significance (tce_bin_oedp_sig) and secondary eclipse check (tce_bin_half_sig) are the most critical discriminators.
- **Not Working:** The deterministic physics baseline suffers from low precision (61.2%) because rigid cutoffs fail on grazing eclipsing binaries with shallow dips and discard valid long-period candidates.
- **Learnings:** False alarms predominantly occur in the low-SNR regime (SNR < 10) where stellar granulation noise mimics shallow transits. Threshold tuning will be necessary to prioritize precision over recall for telescope follow-ups.

#### 6. Challenges & Remaining Plan
1. Train remaining classical models: SVM (RBF kernel), Gradient Boosting (XGBoost/LightGBM), and a tabular MLP.
2. Build and train the PyTorch 1D CNN using dual global/local phase-folded representations (AstroNet architecture).
3. Benchmark all 8 models on the final held-out test set and analyze error breakdown across SNR regimes.

#### 7. Individual Contributions
- **Jackson Lusk:** Ingested the DR24 TCE catalog, mapped target labels, dropped leaky av_* columns, implemented the leak-free GroupShuffleSplit by kepid with automated assertions, and saved clean train/val/test datasets.
- **Taylor Tran:** Built the lightkurve downloading and phase-folding pipeline, generated the PC vs. Eclipsing Binary comparison (Visualization 1), and implemented/scored the deterministic physics baseline.
- **Riley Marek:** Generated feature distribution boxplots (Visualization 2), built the class-weighted Logistic Regression and Random Forest models, generated feature importance charts, compiled the tri-method validation comparison table, and conducted initial error analysis.
```


---
## 6. Phase 2: Advanced Modeling & Deep Learning (Post-Midpoint)
**Timeline:** Weeks 12–14  
**Scope:** Train remaining models, build PyTorch 1D CNN, and execute final blind benchmarking on `test.csv`.


In [ ]:
# [POST-MIDPOINT: RILEY] 6.1 Remaining Classical ML Models
# Placeholder for Week 12:
# - Support Vector Machine (RBF Kernel) with StandardScaler
# - Gradient Boosting Classifier (XGBoost / LightGBM)
# - Tabular Multi-Layer Perceptron (MLPClassifier)
# - Hyperparameter optimization via 5-Fold StratifiedGroupKFold on train+val



In [ ]:
# [POST-MIDPOINT: TAYLOR] 6.2 1D Convolutional Neural Network (PyTorch)
# Placeholder for Weeks 12-13:
# - AstroNet Dual-Branch 1D CNN Architecture:
#     Branch 1: Global View (2001 bins, full orbital phase)
#     Branch 2: Local View (201 bins, zoomed transit center)
# - Concatenation -> Dense Layers -> Sigmoid output P(Planet Candidate)
# - Data augmentation: transit jitter and Gaussian noise injection



In [ ]:
# [POST-MIDPOINT: COLLABORATIVE] 6.3 Final Blind Evaluation on test.csv
# Placeholder for Week 14:
# - Evaluate all 8 models on unseen held-out test.csv
# - Comprehensive PR-AUC, ROC-AUC, F1, and Precision/Recall benchmarking
# - Deep dive error analysis: grazing binaries vs. low-SNR noise limits

